In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ══════════════════════════════════════════════════════════════════════
import os
import re
import json
import time
from pathlib import Path

import numpy as np
import chromadb
from huggingface_hub import InferenceClient
from openai import OpenAI
from dotenv import load_dotenv

dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

c:\Users\hp\Desktop\All\college\Fourth_year\Second_term\GP2\GP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — API KEYS & CLIENTS
# ══════════════════════════════════════════════════════════════════════
HF_KEY     = os.getenv("HF_TOKEN")
GITHUB_KEY = os.getenv("GITHUB_TOKEN")

if not HF_KEY:
    raise ValueError("HF_TOKEN is missing. Add it to .env and re-run.")
if not GITHUB_KEY:
    raise ValueError("GITHUB_TOKEN is missing. Add it to .env and re-run.")

hf_client      = InferenceClient(provider="hf-inference", api_key=HF_KEY)
github_client  = OpenAI(base_url="https://models.github.ai/inference", api_key=GITHUB_KEY)

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — EMBEDDING FUNCTION (same as test2.ipynb — BGE-M3)
# ══════════════════════════════════════════════════════════════════════
def embed_texts(texts: list[str]) -> list[list[float]]:
    embeddings = []
    for text in texts:
        vector = np.array(hf_client.feature_extraction(text, model="BAAI/bge-m3"))
        if vector.ndim > 1:
            vector = vector.squeeze()
        vector = vector / np.linalg.norm(vector)
        embeddings.append(vector.tolist())
        time.sleep(0.1)   # avoid hitting rate limits on free tier
    return embeddings

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — CONNECT TO EXISTING CHROMADB (read-only for this evaluation)
# ══════════════════════════════════════════════════════════════════════
DB_PATH = './chroma_db'   # same path used in test2.ipynb
client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection("ucas_knowledge_base")

print(f"Connected. Total chunks in collection: {collection.count()}")

Connected. Total chunks in collection: 284


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — METADATA FILTER DETECTION (copied from test2.ipynb, unchanged)
# ══════════════════════════════════════════════════════════════════════
KNOWN_PROGRAMS = [
    "علم البيانات والذكاء الاصطناعي",
    "هندسة أمن المعلومات السيبراني",
    "شبكات الحاسوب والإنترنت",
    "صيانة الأجهزة الذكية",
    "أمن المعلومات"
]

KNOWN_COURSES = [
    "قرآن كريم", "اللغة الإنجليزية", "أساسيات علم البيانات والذكاء الاصطناعي",
    "لغة برمجة (عملي)", "مقدمة في الحوسبة (عملي)", "لغة برمجة",
    "مقدمة في الحوسبة", "تفاضل وتكامل 1", "تفاضل وتكامل 2",
    "دراسات في السيرة", "لغة إنجليزية تخصصية", "تراكيب بيانات",
    "لغات برمجة علم البيانات (عملي)", "الرياضيات المنفصلة",
    "لغات برمجة علم البيانات", "مبادئ الإحصاء والاحتمالات", "الجبر الخطي",
    "مقدمة في قواعد البيانات (عملي)", "برمجة الذكاء الاصطناعي (عملي)",
    "تحليل وتمثيل البيانات", "مقدمة في قواعد البيانات", "برمجة للذكاء الاصطناعي",
    "تحليل وتمثيل البيانات (عملي)", "التنقيب عن البيانات (عملي)",
    "معمارية الحاسوب", "مبادئ الشبكات", "برمجة تعلم الآلة",
    "تصميم وتحليل الخوارزميات", "التنقيب عن البيانات", "برمجة تعلم الآلة (عملي)",
    "اللغة العربية", "النظم المغموسة (عملي)", "نظم التشغيل", "إنترنت الأشياء",
    "الحوسبة السحابية", "تمييز الأنماط", "النظم المغموسة", "العمل الحر",
    "التعلم العميق (عملي)", "مناهج البحث العلمي والكتابة العلمية",
    "مخازن البيانات", "معالجة الصور الرقمية", "التعلم العميق",
    "الأنظمة الخبيرة", "ريادة الأعمال", "معالجة اللغات الطبيعية (عملي)",
    "أخلاقيات الذكاء الاصطناعي", "برمجة الروبوت (التعليم المعزز)",
    "برمجة الروبوت (التعليم المعزز) عملي", "عمليات تعلم الآلة (MLOps)",
    "معالجة اللغات الطبيعية", "مشروع تخرج (2)", "متطلب تخصص اختياري",
    "دراسات في العقيدة", "هندسة برمجيات", "التدريب الميداني",
    "البيانات الكبيرة", "استرجاع المعلومات", "علم الإدراك والمعرفة",
    "برمجة الروبوت"
]

_FILTER_SYSTEM = f"""
You are an assistant who determines whether a student's question refers to a specific academic program or course.

List of available programs:
{chr(10).join(f'- {p}' for p in KNOWN_PROGRAMS)}

Your task:
1. Extract ALL programs mentioned or clearly referred to. Return their exact names from the list {KNOWN_PROGRAMS}.
2. Extract ALL course codes mentioned (e.g. DSAI1301). Return them as a list.
3. Extract ALL course names mentioned or clearly referred to. Return their exact names from the list {KNOWN_COURSES}.
4. If no specific program is mentioned, return "علم البيانات والذكاء الاصطناعي".
5. If no specific course is mentioned, return null.
6. Determine if the student is asking about the full study plan / curriculum / semester structure of a program. If so, set "is_study_plan" to true.

Return ONLY JSON:
{{"programs": [...] or ["علم البيانات والذكاء الاصطناعي"], "course_codes": [...] or null, "course_names": [...] or null, "is_study_plan": true or false}}
"""

def detect_metadata_filter(query: str) -> tuple[dict | None, bool]:
    """Ask LLM to detect program/course mentioned in query."""
    try:
        resp = github_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": _FILTER_SYSTEM},
                {"role": "user",   "content": query}
            ],
            temperature=0.0,
            max_tokens=80
        )
        raw = re.sub(r"```json|```", "", resp.choices[0].message.content).strip()
        detected = json.loads(raw)

        raw_programs = detected.get('programs') or []
        if isinstance(raw_programs, str):
            raw_programs = [raw_programs]
        programs = [p for p in raw_programs if p in KNOWN_PROGRAMS]

        course_codes = detected.get("course_codes") or []

        raw_course_names = detected.get("course_names") or []
        if isinstance(raw_course_names, str):
            raw_course_names = [raw_course_names]
        course_names = [p for p in raw_course_names if p in KNOWN_COURSES]

        is_study_plan = detected.get("is_study_plan", False)

        if is_study_plan and programs:
            if len(programs) > 1:
                f = {"$and": [{"category": "study_plan"}, {"program": {"$in": programs}}]}
            else:
                f = {"$and": [{"category": "study_plan"}, {"program": programs[0]}]}
            return f, True

        if course_codes:
            f = {"course_code": {"$in": course_codes}} if len(course_codes) > 1 else {"course_code": course_codes[0]}
            if programs:
                prog_f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
                f = {"$and": [f, prog_f]}
            return f, False

        if course_names:
            f = {"course_name": {"$in": course_names}} if len(course_names) > 1 else {"course_name": course_names[0]}
            if programs:
                prog_f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
                f = {"$and": [f, prog_f]}
            return f, False

        if programs:
            f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
            return f, False

        return None, False

    except Exception as e:
        print(f"[Filter detection error] {e}")
        return None, False

In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — RAW SCORE RETRIEVAL (threshold-agnostic version of search())
# ══════════════════════════════════════════════════════════════════════
def get_raw_score(query: str, top_k: int = 3) -> dict:
    """
    Runs the same retrieval path as production search()/multi_query_search()
    (metadata filter + embedding search) but WITHOUT applying any fixed
    SIMILARITY_THRESHOLD, so we can sweep multiple threshold values later
    against the true best_score.
    """
    metadata_filter, is_study_plan = detect_metadata_filter(query)

    # Study plan queries bypass embedding search entirely (exact get())
    if is_study_plan and metadata_filter:
        results = collection.get(where=metadata_filter)
        has_docs = bool(results["documents"])
        return {"query": query, "best_score": 1.0 if has_docs else 0.0, "is_study_plan": True}

    query_embedding = embed_texts([query])[0]
    query_params = dict(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    if metadata_filter:
        query_params["where"] = metadata_filter

    results   = collection.query(**query_params)
    documents = results["documents"][0]
    distances = results["distances"][0]

    # Guard: metadata filter matched nothing — retry without it
    if not documents and metadata_filter:
        query_params.pop("where", None)
        results   = collection.query(**query_params)
        documents = results["documents"][0]
        distances = results["distances"][0]

    if not documents:
        return {"query": query, "best_score": 0.0, "is_study_plan": False}

    similarities = [1 - d for d in distances]
    return {"query": query, "best_score": max(similarities), "is_study_plan": False}

In [11]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — LABELED TEST SET
# expected_has_answer = True  -> in-scope, should be answered from the KB
# expected_has_answer = False -> out-of-scope, should trigger fallback
# ══════════════════════════════════════════════════════════════════════
test_set = [
    # ── In-scope questions (should be answered) ──────────────────────────
    ("ما هي متطلبات القبول في تخصص علم البيانات؟", True),
    ("ما هو محتوى مساق تحليل وتمثيل البيانات؟", True),
    ("هل يوجد منح دراسية لتخصص الذكاء الاصطناعي؟", True),
    ("من هم أعضاء هيئة التدريس في تخصص علم البيانات؟", True),
    ("ما هي فرص العمل بعد التخرج من تخصص أمن المعلومات؟", True),
    ("كم عدد الساعات المعتمدة لتخصص هندسة الحاسوب؟", True),
    ("ما الفرق بين تخصص علم البيانات وتخصص أمن المعلومات؟", True),
    ("ما هي الخطة الدراسية لتخصص هندسة الذكاء الاصطناعي؟", True),
    ("من هو مدرس مساق تحليل وتمثيل البيانات؟", True),
    ("ما هو مساق تراكيب بيانات؟", True),

    # ── Out-of-scope questions (should trigger fallback) ─────────────────
    ("ما هو تصنيف جامعة UCAS مقارنة بجامعة النجاح؟", False),
    ("متى موعد امتحانات نهاية الفصل الدراسي القادم؟", False),
    ("هل هذا التخصص يعتبر ضمن النادي الهندسي؟", False),
    ("هل يتم تدريس هذا التخصص في جامعات أخرى؟", False),
    ("ما التخصصات المشابهة في الجامعة الإسلامية؟", False),
    ("هل تقدم الجامعة الأمريكية في القاهرة نفس التخصص؟", False),
]

print(f"Test set size: {len(test_set)}  "
      f"(in-scope: {sum(1 for _, e in test_set if e)}, "
      f"out-of-scope: {sum(1 for _, e in test_set if not e)})")

Test set size: 16  (in-scope: 10, out-of-scope: 6)


In [15]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8 — STEP 1: Compute raw score for every query ONCE
# (expensive part — one embedding + one LLM filter call per query)
# ══════════════════════════════════════════════════════════════════════
print("Computing raw similarity scores for the test set...\n")

raw_results = []
for query, expected in test_set:
    r = get_raw_score(query)
    raw_results.append({"query": query, "expected": expected, "score": r["best_score"]})
    tag = "IN-SCOPE " if expected else "OUT-SCOPE"
    print(f"[{tag}] score={r['best_score']:.4f}  {query}")

Computing raw similarity scores for the test set...

[IN-SCOPE ] score=0.6843  ما هي متطلبات القبول في تخصص علم البيانات؟
[IN-SCOPE ] score=0.6188  ما هو محتوى مساق تحليل وتمثيل البيانات؟
[IN-SCOPE ] score=0.6638  هل يوجد منح دراسية لتخصص الذكاء الاصطناعي؟
[IN-SCOPE ] score=0.6475  من هم أعضاء هيئة التدريس في تخصص علم البيانات؟
[IN-SCOPE ] score=0.6683  ما هي فرص العمل بعد التخرج من تخصص أمن المعلومات؟
[IN-SCOPE ] score=0.7208  كم عدد الساعات المعتمدة لتخصص هندسة الحاسوب؟
[IN-SCOPE ] score=0.6286  ما الفرق بين تخصص علم البيانات وتخصص أمن المعلومات؟
[IN-SCOPE ] score=1.0000  ما هي الخطة الدراسية لتخصص هندسة الذكاء الاصطناعي؟
[IN-SCOPE ] score=0.5819  من هو مدرس مساق تحليل وتمثيل البيانات؟
[IN-SCOPE ] score=0.5794  ما هو مساق تراكيب بيانات؟
[OUT-SCOPE] score=0.3740  ما هو تصنيف جامعة UCAS مقارنة بجامعة النجاح؟
[OUT-SCOPE] score=0.6192  متى موعد امتحانات نهاية الفصل الدراسي القادم؟
[OUT-SCOPE] score=0.5873  هل هذا التخصص يعتبر ضمن النادي الهندسي؟
[OUT-SCOPE] score=0.5031  هل يتم تدريس هذا

In [16]:
# ══════════════════════════════════════════════════════════════════════
# CELL 9 — STEP 2: Sweep candidate thresholds against pre-computed scores
# ══════════════════════════════════════════════════════════════════════
candidate_thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

print(f"{'Threshold':<10}{'TP':<5}{'FP':<5}{'TN':<5}{'FN':<5}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}")

sweep_results  = []
best_threshold = None
best_accuracy  = -1

for t in candidate_thresholds:
    tp = fp = tn = fn = 0
    for r in raw_results:
        predicted, actual = r["score"] >= t, r["expected"]
        if predicted and actual:       tp += 1
        elif predicted and not actual: fp += 1
        elif not predicted and actual: fn += 1
        else:                          tn += 1

    total     = tp + fp + tn + fn
    accuracy  = (tp + tn) / total
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    sweep_results.append({
        "threshold": t, "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "accuracy": accuracy, "precision": precision, "recall": recall
    })

    print(f"{t:<10.2f}{tp:<5}{fp:<5}{tn:<5}{fn:<5}{accuracy:<10.2%}{precision:<10.2%}{recall:<10.2%}")

    if accuracy > best_accuracy:
        best_accuracy, best_threshold = accuracy, t

print(f"\nBest threshold by accuracy: {best_threshold}  (accuracy={best_accuracy:.2%})")

Threshold TP   FP   TN   FN   Accuracy  Precision Recall    
0.30      10   6    0    0    62.50%    62.50%    100.00%   
0.35      10   6    0    0    62.50%    62.50%    100.00%   
0.40      10   5    1    0    68.75%    66.67%    100.00%   
0.45      10   5    1    0    68.75%    66.67%    100.00%   
0.50      10   4    2    0    75.00%    71.43%    100.00%   
0.55      10   2    4    0    87.50%    83.33%    100.00%   
0.60      8    1    5    2    81.25%    88.89%    80.00%    
0.65      5    0    6    5    68.75%    100.00%   50.00%    
0.70      2    0    6    8    50.00%    100.00%   20.00%    
0.75      1    0    6    9    43.75%    100.00%   10.00%    
0.80      1    0    6    9    43.75%    100.00%   10.00%    

Best threshold by accuracy: 0.55  (accuracy=87.50%)


In [17]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 (OPTIONAL) — Zero-false-positive threshold (safer choice)
# Picks the LOWEST threshold that produces zero false positives,
# since a false positive (answering from irrelevant context) is riskier
# than a false negative (an unnecessary fallback to the advisor).
# ══════════════════════════════════════════════════════════════════════
zero_fp_thresholds = [r for r in sweep_results if r["fp"] == 0]

if zero_fp_thresholds:
    safest = min(zero_fp_thresholds, key=lambda r: r["threshold"])
    print(f"Lowest threshold with zero false positives: {safest['threshold']}")
    print(f"  -> accuracy={safest['accuracy']:.2%}, "
          f"precision={safest['precision']:.2%}, recall={safest['recall']:.2%}, "
          f"FN={safest['fn']}")
else:
    print("No candidate threshold in the tested range eliminates all false positives.")

Lowest threshold with zero false positives: 0.65
  -> accuracy=68.75%, precision=100.00%, recall=50.00%, FN=5
